In [0]:
# Setup inicial e criação do schema Bronze
import requests
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# Criar schema Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

DataFrame[]

In [0]:
# Definição do caminho do Volume e tabelas de destino

input_path = "/Volumes/workspace/default/input"

files_tables = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics", 
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews"
}

In [0]:
# Ingestão dos CSVs na camada Bronze

for file_name, table_name in files_tables.items():
    file_path = f"{input_path}/{file_name}"

    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(file_path)
    )

    # timestamp inserção
    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    # Gravação Delta em modo Append
    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )




In [0]:
# Widgets para parâmetros de data (formato MM-DD-AAAA)
try:
    dbutils.widgets.remove("data_inicio")
    dbutils.widgets.remove("data_fim")
except:
    pass

# Consulta últimos 7 dias corridos
data_fim = datetime.now().strftime("%m-%d-%Y")
data_inicio= (datetime.now() - timedelta(days=7)).strftime("%m-%d-%Y")

dbutils.widgets.text("data_inicio", data_inicio, "Data Início")
dbutils.widgets.text("data_fim", data_fim, "Data Fim")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

# Endpoint PTAX
url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    f"&$format=json"
)

response = requests.get(url)

# Converte a resposta para JSON
data = response.json()

# Verifica se a API retornou cotações
if data["value"]:

    # Converte os dados da API em DataFrame
    df_cotacao = spark.createDataFrame(data["value"])

    # Adiciona a data e hora da ingestão
    df_cotacao = df_cotacao.withColumn(
        "ingestion_datetime",
        F.current_timestamp()
    )

    # Salva na Bronze em formato Delta
    (
        df_cotacao.write
        .format("delta")
        .mode("append")
        .saveAsTable("bronze.tb_cotacao_dolar")
    )

    display(df_cotacao)

else:
    print("Nenhuma cotação retornada para o período.")

cotacaoCompra,cotacaoVenda,dataHoraCotacao,ingestion_datetime
5.1803,5.1809,2026-09-30 13:11:44.783262,2026-10-07T20:41:14.972Z
5.2073,5.2079,2026-10-01 13:10:35.4469,2026-10-07T20:41:14.972Z
5.2232,5.2238,2026-10-02 13:03:16.256632,2026-10-07T20:41:14.972Z
4.9853,4.9859,2026-10-05 13:07:36.558602,2026-10-07T20:41:14.972Z
4.9692,4.9698,2026-10-06 13:03:21.267287,2026-10-07T20:41:14.972Z
4.9929,4.9935,2026-10-07 13:05:23.249063,2026-10-07T20:41:14.972Z
